## Demodulación FSK de Sunde (M=2)

In [ ]:
# Importamos las librerías necesarias
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Se definen parámetros

# Probabilidad de los bits
p0 = 0.5
p1 = 1 - p0

# Número de bits de la secuencia a generar
N = 50000000  # 50 millones de bits

# Parámetros de modulación 2-FSK de Sunde
formato = 'polar'
M = 2
n = int(np.log2(M))
Tb = 0.001
fb = 1/Tb

# Parámetros para la señal modulada
fmuestreo = 100000
fc = 2*fb  # 2000 Hz para FSK de Sunde
Ac = 1

In [ ]:
def generador_secuencia(n, p0, p1, semilla=None):
    # Validación
    if not np.isclose(p0 + p1, 1):
        raise ValueError("Las probabilidades deben cumplir: p0 + p1 = 1")
    rng = np.random.default_rng(semilla)
    return rng.choice([0, 1], size=n, p=[p0, p1])

# Generar secuencia
secuencia = generador_secuencia(N, p0, p1, semilla=42)
print(f"Secuencia generada (primeros 20 bits): {secuencia[:20]}")

In [ ]:
def gray_code(n):
    """Genera códigos Gray de n bits"""
    if n == 0:
        return ['']
    prev = gray_code(n-1)
    return ['0' + code for code in prev] + ['1' + code for code in reversed(prev)]

In [ ]:
# Convertidor de datos
def convertidor_datos(bits, M=2, formato='polar'):
    n = int(np.log2(M))
    
    # Asegurar múltiplo de n
    L = len(bits)
    if L % n != 0:
        bits = bits[:L - (L % n)]
    
    # Agrupar bits (para M=2, n=1, cada bit es un símbolo)
    grupos = bits.reshape(-1, n)
    
    # Generar código Gray
    gray = gray_code(n)
    
    # Crear tabla de lookup vectorizada
    gray_array = np.array([[int(b) for b in codigo] for codigo in gray])
    
    # Convertir grupos a índices usando broadcasting
    indices = np.where((grupos[:, None, :] == gray_array[None, :, :]).all(axis=2))[1]
    
    # Niveles 2-FSK (formato polar: -1, +1)
    if formato == 'polar':
        niveles = 2*indices - (M - 1)
    elif formato == 'unipolar':
        niveles = indices
    else:
        raise ValueError("Formato debe ser 'polar' o 'unipolar'")
    
    return niveles

bk = convertidor_datos(secuencia, M, formato)
print(f"Símbolos bk (primeros 20): {bk[:20]}")

In [ ]:
# Calcular probabilidades de los símbolos
gray = gray_code(n)
p_simbolos = []
for codigo in gray:
    prob = 1.0
    for bit in codigo:
        if bit == '0':
            prob *= p0
        else:
            prob *= p1
    p_simbolos.append(prob)
p_simbolos = np.array(p_simbolos)

# Generar niveles según el formato
indices = np.arange(M)
if formato == 'polar':
    niveles_fsk = 2*indices - (M - 1)
elif formato == 'unipolar':
    niveles_fsk = indices

print(f"Formato: {formato}")
print(f"Suma de probabilidades: {np.sum(p_simbolos):.6f}")
print(f"\nProbabilidades de cada símbolo:")
for i, (nivel, prob) in enumerate(zip(niveles_fsk, p_simbolos)):
    print(f"  b_k = {nivel:3d}, código Gray = {gray[i]}, p(b_k) = {prob:.6f}")

In [ ]:
# Cálculo de la media de Qk (componente en cuadratura)
# Para FSK de Sunde: Qk = (-1)^k * bk
def get_media_qk(p_simbolos, niveles):
    # Media de Qk considerando la alternancia (-1)^k
    # Para secuencias largas, la alternancia promedia a 0
    # pero consideramos la magnitud promedio
    media = np.sum(niveles * p_simbolos)
    return media

media_qk = get_media_qk(p_simbolos, niveles_fsk)
print(f"Media de Qk: {media_qk}")

# Varianza de Qk
def get_varianza_qk(p_simbolos, niveles, media):
    var = np.sum((niveles**2) * p_simbolos) - media**2
    return var

var_qk = get_varianza_qk(p_simbolos, niveles_fsk, media_qk)
print(f"Varianza de Qk: {var_qk}")

In [ ]:
# Generación de ruido AWGN para diferentes SNR
# Para FSK de Sunde, la energía por bit es Eb = Ac²*Tb/2
Eb = (Ac**2 * Tb) / 2

# Rango de SNR por bit en dB
gamma_b_dB = np.arange(5, 31, 1)

# Diccionario para almacenar resultados
resultados = {}

for snr_db in gamma_b_dB:
    # Convertir SNR de dB a escala lineal
    gamma_b_lineal = 10**(snr_db / 10)
    
    # Densidad espectral de potencia del ruido
    N0 = Eb / gamma_b_lineal
    
    # Varianza del ruido para componentes I y Q
    # σ² = N0/2 (para cada componente)
    sigma2 = N0 / 2
    sigma = np.sqrt(sigma2)
    
    # Generar ruido para componentes I y Q
    ni = np.random.normal(0, sigma, size=len(bk))
    nq = np.random.normal(0, sigma, size=len(bk))
    
    resultados[snr_db] = {
        'ni': ni,
        'nq': nq,
        'sigma2': sigma2,
        'sigma': sigma
    }

print(f"\nRuido generado para SNR de {gamma_b_dB[0]} dB a {gamma_b_dB[-1]} dB")
print(f"Ejemplo para SNR = 10 dB:")
print(f"  σ² = {resultados[10]['sigma2']:.6f}")
print(f"  σ = {resultados[10]['sigma']:.6f}")

In [ ]:
# Gráfica de la fdp del ruido para SNR = 10 dB
snr_plot = 10
ni_plot = resultados[snr_plot]['ni']
sigma_plot = resultados[snr_plot]['sigma']

x = np.linspace(ni_plot.min(), ni_plot.max(), 1000)
fdp_teorica = (1/(sigma_plot * np.sqrt(2*np.pi))) * np.exp(-x**2 / (2*sigma_plot**2))

plt.figure(figsize=(10, 5))
plt.hist(ni_plot, bins=500, density=True, alpha=0.6, color='blue', edgecolor='blue', label='Histograma del ruido')
plt.plot(x, fdp_teorica, 'r-', linewidth=2, label=f'fdp teórica N(0, σ²={sigma_plot**2:.4f})')
plt.xlabel('Amplitud del ruido')
plt.ylabel('Densidad de probabilidad')
plt.title(f'Función de densidad de probabilidad del ruido - SNR = {snr_plot} dB')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Demodulación FSK de Sunde
# Las señales recibidas después del canal AWGN son:
# Componente I (en fase): Ik = Ac + ni
# Componente Q (cuadratura): Qk = (-1)^k * Ac * bk + nq

for snr_db in gamma_b_dB:
    ni = resultados[snr_db]['ni']
    nq = resultados[snr_db]['nq']
    
    # Componente I recibida (constante + ruido)
    Ik_rx = Ac + ni
    
    # Componente Q recibida (modulada + ruido)
    # Qk = (-1)^k * bk + nq (considerando Ac=1)
    k_indices = np.arange(len(bk))
    Qk_rx = ((-1)**k_indices) * Ac * bk + nq
    
    resultados[snr_db]['Ik_rx'] = Ik_rx
    resultados[snr_db]['Qk_rx'] = Qk_rx

print(f"Ejemplo para SNR = 10 dB:")
print(f"Símbolos transmitidos bk (primeros 5): {bk[:5]}")
print(f"Componente I recibida (primeros 5): {resultados[10]['Ik_rx'][:5]}")
print(f"Componente Q recibida (primeros 5): {resultados[10]['Qk_rx'][:5]}")

In [ ]:
# Detector de símbolos FSK
# Para FSK de Sunde, la decisión se basa en el signo de Qk*(-1)^k
# Si Qk*(-1)^k > 0 → bk = +1 (bit 1)
# Si Qk*(-1)^k < 0 → bk = -1 (bit 0)

for snr_db in gamma_b_dB:
    Qk_rx = resultados[snr_db]['Qk_rx']
    k_indices = np.arange(len(bk))
    
    # Demodular eliminando la alternancia (-1)^k
    decision = Qk_rx * ((-1)**k_indices)
    
    # Estimación de símbolos
    bk_estimado = np.where(decision > 0, 1, -1)
    
    resultados[snr_db]['bk_est'] = bk_estimado

print(f"Ejemplo para SNR = 10 dB:")
print(f"bk transmitido (primeros 20):  {bk[:20]}")
print(f"bk estimado (primeros 20):     {resultados[10]['bk_est'][:20]}")
print(f"Errores de símbolos: {np.sum(bk != resultados[10]['bk_est'])} de {len(bk)} símbolos")

In [ ]:
# Demapeador: símbolos a bits
def demapeador_simbolos(simbolos, M=2, formato='polar'):
    n = int(np.log2(M))
    gray = gray_code(n)
    
    # Convertir símbolos a índices
    if formato == 'polar':
        indices = ((simbolos + (M - 1)) / 2).astype(int)
    elif formato == 'unipolar':
        indices = simbolos.astype(int)
    else:
        raise ValueError("Formato debe ser 'polar' o 'unipolar'")
    
    # Crear tabla de lookup
    lookup_table = np.array([[int(b) for b in codigo] for codigo in gray], dtype=np.int8)
    
    # Procesar por chunks
    chunk_size = 1000000
    num_chunks = (len(indices) + chunk_size - 1) // chunk_size
    
    bits_estimados = np.empty(len(indices) * n, dtype=np.int8)
    
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(indices))
        chunk_indices = indices[start:end]
        
        bits_chunk = lookup_table[chunk_indices].flatten()
        bits_estimados[start*n:end*n] = bits_chunk
    
    return bits_estimados

for snr_db in gamma_b_dB:
    bk_est = resultados[snr_db]['bk_est']
    bits_estimados = demapeador_simbolos(bk_est, M, formato)
    resultados[snr_db]['bits_est'] = bits_estimados

print(f"Ejemplo para SNR = 10 dB:")
print(f"Bits transmitidos (primeros 20): {secuencia[:20]}")
print(f"Bits estimados (primeros 20):    {resultados[10]['bits_est'][:20]}")
print(f"Errores de bits: {np.sum(secuencia[:len(resultados[10]['bits_est'])] != resultados[10]['bits_est'])} de {len(resultados[10]['bits_est'])} bits")

In [ ]:
# Cálculo y gráfica del BER
from scipy.special import erfc

BER = []
bits_tx = secuencia[:len(bk)*n]

for snr_db in gamma_b_dB:
    bits_est = resultados[snr_db]['bits_est']
    errores = np.sum(bits_tx != bits_est)
    ber = errores / len(bits_tx)
    BER.append(ber)
    resultados[snr_db]['BER'] = ber

print(f"# de bits estimados: {len(bits_est)}")
BER = np.array(BER)

# BER teórico para FSK ortogonal coherente
# Pb = 0.5 * erfc(sqrt(Eb/N0))
BER_teorico = []

for snr_db in gamma_b_dB:
    gamma_b_lin = 10**(snr_db / 10)
    pb_teorico = 0.5 * erfc(np.sqrt(gamma_b_lin))
    BER_teorico.append(pb_teorico)

BER_teorico = np.array(BER_teorico)

plt.figure(figsize=(10, 6))
plt.semilogy(gamma_b_dB, BER, 'bo-', linewidth=2, markersize=8, label='BER simulado')
plt.semilogy(gamma_b_dB, BER_teorico, 'r--', linewidth=2, label='BER teórico')
plt.xlabel('SNR por bit (dB)')
plt.ylabel('BER')
plt.title(f'Tasa de Error de Bit (BER) vs SNR - {M}-FSK de Sunde (formato={formato})')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.xlim([gamma_b_dB[0], gamma_b_dB[-1]])
plt.show()

print(f"\nResultados BER:")
print(f"{'SNR (dB)':<10} {'BER simulado':<15} {'BER teórico':<15}")
for snr_db, ber_sim, ber_teo in zip(gamma_b_dB, BER, BER_teorico):
    print(f"{snr_db:<10d} {ber_sim:<15.6e} {ber_teo:<15.6e}")

## CONSTELACIÓN

In [ ]:
import plotly.graph_objects as go

snr_db = 13
Ik_rx = resultados[snr_db]['Ik_rx']
Qk_rx = resultados[snr_db]['Qk_rx']

# Parámetros de bins
num_bins_x = 200
num_bins_y = 200

# Histograma 2D
hist_3d, xe_3d, ye_3d = np.histogram2d(Ik_rx, Qk_rx, bins=[num_bins_x, num_bins_y], density=True)

# Centros de bins
x_centers = (xe_3d[:-1] + xe_3d[1:]) / 2
y_centers = (ye_3d[:-1] + ye_3d[1:]) / 2

X_mesh, Y_mesh = np.meshgrid(x_centers, y_centers)

fig_bars = go.Figure()

# Aplanar para iterar
x_flat = []
y_flat = []
z_flat = []
colors_flat = []

# Normalizar para colores
z_max = hist_3d.max()
z_min = hist_3d[hist_3d > 0].min() if np.any(hist_3d > 0) else 0

for i in range(num_bins_x):
    for j in range(num_bins_y):
        if hist_3d[i, j] > 0:
            x_flat.extend([x_centers[i], x_centers[i], None])
            y_flat.extend([y_centers[j], y_centers[j], None])
            z_flat.extend([0, hist_3d[i, j], None])
            colors_flat.extend([hist_3d[i, j], hist_3d[i, j], hist_3d[i, j]])

fig_bars.add_trace(go.Scatter3d(
    x=x_flat,
    y=y_flat,
    z=z_flat,
    mode='lines',
    line=dict(
        color=colors_flat,
        colorscale='Turbo',
        width=3,
        cmin=z_min,
        cmax=z_max,
        colorbar=dict(title='Probabilidad')
    ),
    hoverinfo='skip'
))

# Layout
fig_bars.update_layout(
    title=f'Histograma 3D - {M}-FSK de Sunde (SNR={snr_db} dB)<br>Bins: {num_bins_x}x{num_bins_y}',
    scene=dict(
        xaxis_title='I (In-Phase)',
        yaxis_title='Q (Quadrature)',
        zaxis_title='Probabilidad',
        camera=dict(
            projection=dict(type='orthographic'),
            eye=dict(x=1.5, y=1.5, z=1.2)
        )),
    width=1000,
    height=800
)

fig_bars.show()

print(f"Histograma 3D creado con {num_bins_x}x{num_bins_y} bins usando Plotly")

In [ ]:
# Histograma 2D (vista desde arriba)
snr_db = 13
Ik_rx = resultados[snr_db]['Ik_rx']
Qk_rx = resultados[snr_db]['Qk_rx']

# Histograma 2D
hist, xe, ye = np.histogram2d(Ik_rx, Qk_rx, bins=500, density=True)

# Centros de bins
x_centers = (xe[:-1] + xe[1:]) / 2
y_centers = (ye[:-1] + ye[1:]) / 2

fig = go.Figure(data=go.Heatmap(
    x=x_centers,
    y=y_centers,
    z=hist.T,
    colorscale='Turbo',
    colorbar=dict(title='Probabilidad')
))

# Layout
fig.update_layout(
    title=f'Histograma 2D - {M}-FSK de Sunde (SNR={snr_db} dB)',
    xaxis_title='I (In-Phase)',
    yaxis_title='Q (Quadrature)',
    width=800,
    height=700
)

fig.show()

In [ ]:
# Diagrama de constelación teórico FSK de Sunde
A = Ac

# Puntos ideales de la constelación
# Bit 0 (bk=-1): I=1, Q=-1 (φ=-45°, f=fc-fb/2)
# Bit 1 (bk=+1): I=1, Q=+1 (φ=+45°, f=fc+fb/2)
puntos_ideal = [(A, -A), (A, A)]
bits = ['0', '1']
frecuencias = [fc - fb/2, fc + fb/2]

fig, ax = plt.subplots(figsize=(10, 10))

# Círculo de referencia
circle = plt.Circle((0, 0), np.sqrt(2)*A, fill=False, color='gray', 
                     linestyle='--', linewidth=2, alpha=0.5)
ax.add_patch(circle)

# Puntos de la constelación
for i, (I, Q) in enumerate(puntos_ideal):
    # Vector desde el origen
    ax.arrow(0, 0, I*0.92, Q*0.92, head_width=0.15, head_length=0.1, 
             fc='green', ec='darkgreen', linewidth=5, zorder=4)
    ax.scatter(I, Q, s=500, c='green', marker='o', 
               edgecolors='darkgreen', linewidths=4, zorder=5)
    
    # Etiqueta
    angle = np.degrees(np.arctan2(Q, I))
    ax.text(I*1.3, Q*1.3, f'Bit {bits[i]}\nφ={angle:.0f}°\nf={frecuencias[i]:.0f} Hz', 
            fontsize=11, weight='bold', ha='center', 
            bbox=dict(boxstyle='round', facecolor='yellow', edgecolor='black', linewidth=2))

# Componentes
ax.arrow(0, 0, A*0.95, 0, head_width=0.12, head_length=0.08, 
         fc='blue', ec='blue', linewidth=4, zorder=2)
ax.text(A/2, 0.2, 'cos(2πfb·t)\n[componente I]', fontsize=10, color='blue', 
        weight='bold', ha='center', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Configuración
ax.axhline(0, color='black', linewidth=1.5, alpha=0.7)
ax.axvline(0, color='black', linewidth=1.5, alpha=0.7)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlabel('In-Phase (I)', fontsize=12, fontweight='bold')
ax.set_ylabel('Quadrature (Q)', fontsize=12, fontweight='bold')
ax.set_title('Diagrama de Constelación 2-FSK de Sunde (Ideal)', fontsize=14, fontweight='bold')
ax.set_xlim([-0.5, 2])
ax.set_ylim([-1.8, 1.8])
ax.set_aspect('equal')

# Info
info = f'I: cos(2πfb·t) [fijo]\nQ: ±sen(2πfb·t) [modula]\nConstelación: ±45°'
ax.text(0.98, 0.98, info, transform=ax.transAxes, fontsize=10, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', linewidth=2))

plt.tight_layout()
plt.show()

print(f"Constelación {M}-FSK de Sunde:")
print(f"  Bit 0: I={A:.2f}, Q={-A:.2f}, φ=-45°, f={fc-fb/2:.0f} Hz")
print(f"  Bit 1: I={A:.2f}, Q={+A:.2f}, φ=+45°, f={fc+fb/2:.0f} Hz")